# Challenge 3—Community Economic Resilience & Micro-Grants

### Load & Explore

Welcome. By the end of this notebook you will have five tables in your own BigQuery project
describing a real commercial corridor as a **network**, you will know exactly which of the
connections in it are measured and which are modelled, and you will have seen the one query
shape that makes this challenge different from a spreadsheet.

**Read the text, not just the code.** This notebook is written so you can follow what is
happening by reading only the narrative between the cells. If you have never used BigQuery
before you will still be able to follow. If you have, skim the prose and run the cells.

---

## What your team is building today

**A case for an intervention.** A community lender or a city economic-development officer has a
small fund—call it fifty thousand dollars—and more applicants than money. The obvious move is to
rank the applicants: who is most at risk, who employs the most people, who serves the neediest
neighbourhood. That produces a list, and a list is what everyone else in this room will build.

Here is what a list cannot see.

> A hardware store closing is not one closure. It is a lumber supplier who loses a buyer, three
> contractors who lose their nearest source of materials, and a lunch counter that loses the
> weekday trade of the crews who used to park there. Some of those businesses are not in
> distress today. They become distressed because of a decision nobody connected to them.

**Your agent's job is to trace that, and to argue for a grant on the strength of what it
protects downstream.** Not "this business is struggling"—anyone can see that. *"This business is
the only supplier to four others within the corridor, and losing it costs thirty-one jobs across
businesses that are currently healthy."*

## Before anything else: which parts of this data are real

You are going to be asked this by a judge, so let us settle it in the first thirty seconds. It
is a more interesting answer than it looks.

**The businesses are real.** Every node in your graph is a real registered business in San
Francisco, with its real name, real street address, real coordinates and real industry code,
published by the City. We do not invent businesses.

**The corridor is real.** You are not drawing a box on a map. San Francisco assigns each
registered business to a named commercial corridor—North Beach, Geary Boulevard, 24th Street—and
we use the City's own boundary.

**Two of the connections are real.** Which bank lent to which business, and which brand a
business is franchised to, both come from the Small Business Administration's loan records. So
does something more valuable: **which of those loans were charged off**, meaning the business
actually failed.

**Two of the connections are modelled, and this is the important sentence:**

> The relationship *types*, and their *industry-level average intensities*, are measured and
> published by the federal government. The assignment of those averages to specific pairs of
> businesses is ours—a modelling choice, not a measurement.

The Bureau of Economic Analysis publishes how much, on average, a restaurant buys from a food
wholesaler. It does **not** publish that *this* café buys from *that* butcher. Section 8 shows
you exactly how we get from the first statement to the second, and what it costs in confidence.

Being able to say that distinction out loud is part of what you are scored on.

## What this notebook does

It gets you a graph-shaped dataset and is honest about where every edge came from. It does
**not** create your property graph and it does **not** build your agent. Those are yours.

Your required differentiating technology is **BigQuery property graphs**, queried with **GQL**.
Section 14 sets up the problem, hands you the DDL shape, and names the traps that will otherwise
cost you forty minutes. It stops there deliberately.

## Where this fits in your day

Your table is eight to ten people, which is too many to have everyone typing into one file. The
work splits into four lanes that run in parallel:

| Lane | What they do | Starts |
|---|---|---|
| **Data** | This notebook, then the property graph and the cascade query | now |
| **Agent** | ADK agent, tools, the MCP server | now |
| **Front end** | The face: `adk web`, a custom UI, or Gemini Enterprise | now, against a mock |
| **Story** | The grant memo, the pitch, the demo | now |

If you are reading this, you are probably the data lane. **Budget about 40 minutes.** The single
most useful thing you can do for your team is finish this and get the property graph created
early, because the agent lane cannot traverse a graph that does not exist yet.

# 1. Setup

## First, two decisions that are not really configuration settings

The cell below asks you to set `CITY` and `CORRIDOR`, and both are worth ten seconds rather than
leaving the defaults.

**This is not an application for a city. It is an application for a few blocks.** That is not a
simplification—it is the actual scale at which this problem exists. A lender deciding between a
bakery and a shoe repair shop is making a decision about one commercial strip, where the
businesses genuinely do share customers, suppliers and foot traffic. Model a whole city and the
connections become so diffuse that the cascade means nothing.

**San Francisco does something unusually helpful here: the City assigns every registered
business to a named commercial corridor**, so you do not have to draw a boundary and defend it.
North Beach, Geary Boulevard, 24th Street, Noriega—these are the City's own designations, and
each one is a few hundred businesses, which is the right size for a graph you can reason about.

**You do not have to pick the corridor we default to.** Pick one somebody on your team can
picture. If two of you know the Mission, build 24th Street.

**Want a city that is not San Francisco?** Los Angeles is also tested—see the `CITY` options
below. Appendix B at the bottom works out whether any other corridor or bounding box is worth
building on before you commit an afternoon to it.

## What else we are about to do

Work out which Google Cloud project we are in, create a dataset to hold our tables, and set up a
timer so you can see where the minutes went.

In [ ]:
# --- Configuration -----------------------------------------------------------

# WHICH CORRIDOR ARE YOU SOLVING FOR?
#
# These are real decisions, not config values - see the markdown above.
#
# San Francisco publishes a `business_corridor` label on every registered
# business, so these are the City's own boundaries rather than ours. The
# bracketed number is active businesses in that corridor, measured from the
# City's API on 2026-08-08.
#
# The first number is what the City publishes. The second, where we have it, is
# what survives the privacy filters in Section 4 - and that is the one that
# decides whether you have a graph. Measured 2026-08-09.
#
#     "Central Market"     [230 businesses, 65 industries, 86 with cascade
#                           depth, 22% poverty. TESTED. The default - it has
#                           the most graph structure AND the highest need]
#     "North Beach"        [148 businesses, 35 industries, only 18 with depth]
#     "24th St"            [473 ->  89, 30 industries. Workable but thin]
#     "Third Street"       [311 ->  36, 19 industries. TOO THIN - do not use]
#
# Not measured by us, but larger and therefore safer. Appendix B measures any of
# them in about a minute, and you should run it before committing an afternoon:
#     "Chinatown" [1909]  "Market/Castro" [1519]  "Central Market" [1367]
#     "Mission Street" [1024]  "Union Street" [885]  "Parkside Taraval" [596]
#     "Geary Boulevard" [479]  "West Portal" [476]
#
# Larger, if you want a busier graph: "Chinatown" [1908], "Market/Castro" [1519],
# "Mission Street" [1024], "Union Street" [885]. Note that above roughly 800
# nodes the graph visualisation in Section 3 stops being readable - see the note
# there about the 2 MB limit.
CITY     = "San Francisco"
CORRIDOR = "Central Market"

DATASET  = "a4i_econ"     # if you change this, also change the %%bigquery cells
                          # that name it literally (Sections 3 and 12)
LOCATION = "US"           # keep every table in the same location

# Los Angeles is the second tested city. It has no corridor column, so you give
# a bounding box instead. Uncomment and set both of these to use it.
#     CITY = "Los Angeles"
#     LA_BBOX = (34.04, 34.08, -118.34, -118.29)   # (lat_min, lat_max, lon_min, lon_max)
LA_BBOX = None

# --- Timing, so you know where your minutes went ------------------------------
import time
NOTEBOOK_START = time.time()
STEP_TIMES = {}

class step:
    """Context manager that times a block and reports it."""
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        self.t0 = time.time()
        return self
    def __exit__(self, *exc):
        elapsed = time.time() - self.t0
        STEP_TIMES[self.name] = elapsed
        print(f"\n[{self.name}] took {elapsed:.1f}s")
        return False

# --- Which project are we in? -------------------------------------------------
import os
import google.auth

credentials, PROJECT_ID = google.auth.default()
if not PROJECT_ID:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT")
if not PROJECT_ID:
    raise RuntimeError(
        "Could not determine your project ID. Set it manually:\n"
        "    PROJECT_ID = 'your-project-id-here'"
    )

CITIES = {
    "San Francisco": {"state": "CA", "state_fips": "06",
                      "socrata": "https://data.sfgov.org/resource/g8m3-pdis.json"},
    "Los Angeles":   {"state": "CA", "state_fips": "06",
                      "socrata": "https://data.lacity.org/resource/6rrh-rzua.json"},
}
if CITY not in CITIES:
    raise ValueError(f"Unknown city {CITY!r}. Choose one of: {sorted(CITIES)}")
if CITY == "Los Angeles" and not LA_BBOX:
    raise ValueError(
        "Los Angeles has no corridor column, so you must set LA_BBOX to a "
        "(lat_min, lat_max, lon_min, lon_max) tuple. Appendix B helps you pick one."
    )

STATE      = CITIES[CITY]["state"]
STATE_FIPS = CITIES[CITY]["state_fips"]
SOCRATA    = CITIES[CITY]["socrata"]

print(f"Project  : {PROJECT_ID}")
print(f"Dataset  : {DATASET} ({LOCATION})")
print(f"City     : {CITY}, {STATE}")
print(f"Corridor : {CORRIDOR if CITY == 'San Francisco' else LA_BBOX}")

if CITY == "San Francisco" and CORRIDOR == "Central Market":
    print()
    print("  NOTE - you are on the default corridor.")
    print("  If every team leaves it, a third of the room demos the same eight")
    print("  blocks, and the first real decision your team was supposed to make")
    print("  got skipped by not making it. Eight tested alternatives are listed")
    print("  above and changing it costs ten seconds.")

Now we create the dataset. This is the one piece of setup that writes anything, and it is safe
to run more than once—`exists_ok=True` means a second run reuses what is already there rather
than failing.

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID)

dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET}")
dataset_ref.location = LOCATION
dataset = client.create_dataset(dataset_ref, exists_ok=True)

print(f"Ready: {dataset.full_dataset_id}  (location: {dataset.location})")

# 2. Start with the wrong answer

## Why we are doing this on purpose

The obvious way to allocate a small relief fund is to rank the applicants. Score each business
on how much trouble it is in and how much good the money would do, sort the list, and fund from
the top until the money runs out.

Let us watch that fail, in about ninety seconds, on a corridor small enough to hold in your head.

Below are twelve businesses on one imaginary block. **These twelve are illustrative—we made them
up to make a point.** Everything from Section 4 onward is real. We are using a toy here because
a lesson you can see is worth more than a lesson buried in five hundred rows.

Each has an obvious "distress score"—the kind of number you would build from revenue trend,
rent burden and time in business. Rank them and fund the top three.

In [ ]:
import pandas as pd

# Twelve businesses on one imaginary block. ILLUSTRATIVE ONLY - invented to make
# a point. Real data starts in Section 4.
toy = pd.DataFrame([
    # id, name,                     naics_label,        jobs, distress
    ("b01", "Peak Hardware",         "hardware store",      4, 0.55),
    ("b02", "Delgado Lumber Supply", "building supply",     9, 0.28),
    ("b03", "Ng Contracting",        "general contractor",  7, 0.31),
    ("b04", "Ruiz Remodel",          "general contractor",  5, 0.24),
    ("b05", "Baytown Builders",      "general contractor", 11, 0.19),
    ("b06", "The Corner Counter",    "lunch counter",       6, 0.71),
    ("b07", "Adler & Finch LLP",     "law office",         38, 0.12),
    ("b08", "Sunset Dry Cleaning",   "dry cleaner",         3, 0.83),
    ("b09", "Marisol Flowers",       "florist",             2, 0.79),
    ("b10", "Grove Street Books",    "bookshop",            3, 0.88),
    ("b11", "Pacific Physio",        "physical therapy",    8, 0.22),
    ("b12", "Kimura Grocery",        "grocery",             7, 0.44),
], columns=["id", "name", "kind", "jobs", "distress"])

print("THE OBVIOUS ANSWER - rank by distress, fund the top three\n")
print(toy.sort_values("distress", ascending=False)
         .head(3)[["name", "kind", "jobs", "distress"]]
         .to_string(index=False))
print("\n(Grove Street Books, Sunset Dry Cleaning, Marisol Flowers)")
print(f"Jobs directly protected: {toy.nlargest(3, 'distress').jobs.sum()}")

## Look at what came back

A bookshop, a dry cleaner and a florist. Eight jobs. Nothing about that answer is *wrong*—those
three really are the most distressed businesses on the block, and if your goal is to keep the
maximum number of individual shutters open tomorrow morning, that is your list.

But notice what the ranking cannot express. **It scored twelve businesses as if they were twelve
independent things.** It has no way to represent the fact that Peak Hardware, sitting in the
middle of the list at 0.55, is the only place three contractors on this block buy materials—or
that those contractors' crews are most of The Corner Counter's weekday lunch trade.

## The shape of the fix

So let us give the same twelve businesses to a structure that *can* hold a relationship, and ask
a question a ranked list cannot answer: **if this business closes, what else is exposed?**

We will load them into BigQuery as two tables—one for the businesses, one for the connections
between them—declare a **property graph** over the top, and traverse it.

---

## Thirty seconds on what a graph actually is

**If you have never used one, read this. If you have, skip to the next cell.**

A property graph is not a new database and it is not a new storage format. **It is a declaration
you lay over ordinary BigQuery tables you already have.** Nothing is copied, nothing moves, and
you can declare several different graphs over the same tables and pay for the storage once.

It needs exactly two kinds of table:

| | What it holds | Our example |
|---|---|---|
| **Node table** | One row per *thing* | `toy_business` — one row per business |
| **Edge table** | One row per *connection*, carrying the id of each end | `toy_depends` — one row per dependency, with a `source_id` and a `target_id` |

That is genuinely all. If you can write those two tables, you can build a graph. The vocabulary
is small:

- **Node** — a thing. A business.
- **Edge** — a connection between two things. Directed: `A → B` is not `B → A`.
- **Label** — the *type* of a node or edge, so you can say "follow only supply relationships."
- **Property** — a column exposed on a node or edge. `jobs`, `distress`, `intensity`.
- **Traversal** — walking from node to node along edges. This is the part SQL cannot do well.

### The one thing that justifies all of it

You could answer *"who buys directly from Peak Hardware?"* with a `JOIN`. One line, no new
technology, and you would be right to.

Now answer *"who is affected within three steps?"* In SQL that is a JOIN, then another JOIN, then
a third — and **you have to know it is three before you write the query.** Change your mind about
the depth and you rewrite it. Ask for "however far the damage travels" and you cannot express it
at all without recursion.

In GQL the depth is a **parameter**:

```
-[:DependsOn]->{1,3}
```

*Follow this relationship one, two or three times.* Change `3` to `5` and you have a different
question, not a different query. **That is the whole reason this technology is in this challenge**
— not because graphs are fashionable, but because "how far does this go" is the actual question a
grant officer is asking, and it is the one shape SQL is worst at.

Everything else — the DDL, the labels, the `TO_JSON`— is machinery in service of that one idea.

In [ ]:
# The connections between those same twelve. Also illustrative.
toy_edges = pd.DataFrame([
    # source, target, kind
    ("b02", "b01", "supplies"),        # lumber supply -> hardware store
    ("b01", "b03", "supplies"),        # hardware -> contractors
    ("b01", "b04", "supplies"),
    ("b01", "b05", "supplies"),
    ("b03", "b06", "footfall"),        # contractor crews -> lunch counter
    ("b04", "b06", "footfall"),
    ("b05", "b06", "footfall"),
    ("b07", "b06", "footfall"),        # law office staff -> lunch counter
    ("b07", "b08", "footfall"),        # law office staff -> dry cleaner
    ("b06", "b12", "supplies"),        # lunch counter buys from grocery
    ("b11", "b09", "footfall"),
], columns=["source_id", "target_id", "kind"])

for name, df in [("toy_business", toy), ("toy_depends", toy_edges)]:
    client.load_table_from_dataframe(
        df, f"{PROJECT_ID}.{DATASET}.{name}",
        job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
    ).result()
    print(f"loaded {name}: {len(df)} rows")

Now the part that is new.

`CREATE PROPERTY GRAPH` does **not** copy your data or build a new database. It is a declaration
layered over tables you already have: *treat this table as nodes, that one as edges, joined on
these columns.* Nothing moves. You can declare several different graphs over the same tables and
pay for the storage once.

Two things in the DDL below are worth reading carefully now, because they are the two places
teams lose time later:

- **`KEY (...)` is supplied inline.** You will read in some places that the underlying tables
  need declared `PRIMARY KEY ... NOT ENFORCED` constraints. They do not, if you name the key here
  instead—and naming it here is simpler, especially over tables you did not create.
- **Name element tables `dataset.table`, unbackticked.** Not `` `project.dataset.table` ``.
  Backticks around a three-part name make it a *single quoted identifier*, so BigQuery names the
  node the entire string and then cannot find it. The error you get is
  `The referenced node table 'toy_business' is not defined in the property graph`, which points
  at the `REFERENCES` line — several lines away from the line that is actually wrong.
- **`REFERENCES` then takes the *unqualified* node name.** It is `REFERENCES toy_business`, not
  `REFERENCES a4i_econ.toy_business`. This is genuinely inconsistent with the `FOREIGN KEY`
  syntax you would use on a table, and together with the point above it is the most common way
  to lose ten minutes here.

### The anatomy, clause by clause

This is the smallest complete property graph you will see today, so it is worth knowing what
every line is doing before you read past it:

| Clause | What it does |
|---|---|
| `CREATE OR REPLACE PROPERTY GRAPH ds.ToyBlock` | Names the graph. `OR REPLACE` makes it safe to re-run while you are experimenting |
| `NODE TABLES ( ... )` | Which tables hold *things* |
| `a4i_econ.toy_business` | The table. **Dataset-qualified, unbackticked** |
| `KEY (id)` | Which column uniquely identifies a node. **You are responsible for it being unique**—BigQuery does not check |
| `LABEL Business` | The *type*, so a query can say `(b:Business)`. Without it you would match everything |
| `PROPERTIES (id, name, kind, jobs, distress)` | Which columns are visible to queries. List only what you need; unlisted columns are not scanned |
| `EDGE TABLES ( ... )` | Which tables hold *connections* |
| `KEY (source_id, target_id)` | The edge's own identity. Composite here, because a pair defines it |
| `SOURCE KEY (source_id) REFERENCES toy_business (id)` | Where the arrow starts: this column points at that node's key |
| `DESTINATION KEY (target_id) REFERENCES toy_business (id)` | Where it ends. Swap these two and every arrow in your graph reverses |
| `LABEL DependsOn PROPERTIES (kind)` | The edge's type and its visible columns |

**Read `SOURCE`/`DESTINATION` carefully.** They define direction, direction defines what
"downstream" means, and a graph built backwards will run perfectly and answer every question
inside out.

In [ ]:
# NOTE the naming, because it is the trap described above and it is easy to
# reintroduce: element tables are written DATASET.TABLE, unbackticked. A
# backticked three-part name `project.dataset.table` is a single quoted
# identifier, so the node ends up named the whole string and REFERENCES cannot
# find it. The dataset-qualified form resolves against your default project,
# which is the project this notebook is already running in.
GRAPH_DDL = f"""
CREATE OR REPLACE PROPERTY GRAPH {DATASET}.ToyBlock
  NODE TABLES (
    {DATASET}.toy_business
      KEY (id)
      LABEL Business PROPERTIES (id, name, kind, jobs, distress)
  )
  EDGE TABLES (
    {DATASET}.toy_depends
      KEY (source_id, target_id)
      SOURCE KEY (source_id) REFERENCES toy_business (id)
      DESTINATION KEY (target_id) REFERENCES toy_business (id)
      LABEL DependsOn PROPERTIES (kind)
  )
"""
client.query(GRAPH_DDL).result()
print("Property graph ToyBlock created.")
print("\nNote what that cost: no rows were read and nothing was copied.")
print("A property graph is metadata over the tables you already have.")

And now the question a list could not answer. Read the query before you run it—it is short, and
every piece of it does something.

```
MATCH p = (seed:Business WHERE seed.name = 'Peak Hardware')
          -[:DependsOn]->{1,3}(affected:Business)
```

`-[:DependsOn]->{1,3}` means **follow the DependsOn relationship one, two or three hops
outward.** That `{1,3}` is the whole point of this technology. In ordinary SQL, one hop is a
JOIN, two hops is a second JOIN, three is a third—and you have to know how many before you write
the query. Here the depth is a parameter, and "everything reachable within three steps" is a
clause rather than a rewrite.

**One rule about the visualisation, and it will bite you if you skip it.** The cell magic draws a
picture only when the query returns a **path** wrapped in `TO_JSON`. If you return plain columns
instead—`RETURN affected.name`—you get an ordinary table, with no error and no warning to tell
you why. Always finish with `RETURN TO_JSON(p) AS paths`.

In [ ]:
%%bigquery --graph
GRAPH a4i_econ.ToyBlock
MATCH p = (seed:Business WHERE seed.name = 'Peak Hardware')
          -[:DependsOn]->{1,3}(affected:Business)
RETURN TO_JSON(p) AS paths

**That picture is the challenge in one image.**

Peak Hardware sat sixth on the distress ranking and would not have been funded. Follow the edges
out of it and you reach three contractors, then the lunch counter those contractors' crews eat
at, then the grocery the lunch counter buys from. **Six nodes and seven edges**—the seed plus
five businesses downstream, carrying **thirty-six jobs** between them—hanging off a node the
ranked list scored as unremarkable.

Click any node and you get its properties and its neighbours. Peak Hardware's own distress score
is 0.55; the five businesses exposed to it average 0.30, which is the point: **the businesses at
risk from this closure do not look at risk.** That is precisely what a ranked list cannot show
you, because they are not on it.

Now, be careful about what that does and does not prove. **It does not prove Peak Hardware is the
right grant.** Maybe those contractors can buy materials two blocks away and the edge is weak.
Maybe the bookshop closing takes a community anchor with it in a way no supply chain captures.
The graph does not decide for you—it makes the downstream consequence *visible* so a human can
argue about it. Turning that visibility into a defensible recommendation is your job today, and
it is where the judging happens.

Two practical notes before we move on to real data:

- **If you got a table instead of a picture**, you are missing `TO_JSON`, or `bigquery-magics`
  needs installing—see Section 14.
- **The visualisation caps at 2 MB of returned data in a notebook.** Twelve nodes is nothing. A
  five-hop traversal across a nine-hundred-business corridor will blow past it and render
  partially. `LIMIT` before `RETURN`, and prefer `ANY SHORTEST`, are how you keep it readable.

## Now change one number, and watch what it buys you

The claim above was that depth is a parameter rather than a rewrite. Do not take that on trust—
run it. The cell below asks the same question at three different depths, changing exactly one
character each time.

Watch the third column especially. **The businesses reached at three hops are not connected to
Peak Hardware in any way a person would notice**—they are two businesses removed, and that is
precisely the kind of exposure a ranked list, or a single `JOIN`, cannot see.

In [ ]:
# The same question at three depths. The ONLY difference between these three
# queries is one character. In SQL each one would be a different query with a
# different number of JOINs.
for depth in (1, 2, 3):
    sql = f"""
    GRAPH {DATASET}.ToyBlock
    MATCH (seed:Business WHERE seed.name = 'Peak Hardware')
          -[:DependsOn]->{{1,{depth}}}(affected:Business)
    RETURN DISTINCT affected.name AS name, affected.jobs AS jobs
    """
    df = client.query(sql).to_dataframe()
    names = ", ".join(sorted(df.name)) if len(df) else "(nobody)"
    print(f"within {depth} hop{'s' if depth > 1 else ' '}: "
          f"{len(df)} businesses, {int(df.jobs.sum()) if len(df) else 0} jobs")
    print(f"            {names}\n")

print("One character. Three different questions about the same corridor.")
print()
print("Note these return plain columns, so they come back as TABLES - which is")
print("correct here, because we want to read names rather than look at a picture.")
print("The moment you want the picture back, you need TO_JSON on a path. That is")
print("not a style preference; it is the difference between a render and a table.")

# 3. The businesses are real, and the City drew the corridor

## Why this dataset

Every business in your graph comes from San Francisco's **Registered Business Locations** file:
every entity registered to do business at a location in the city, with its trade name, legal
name, street address, coordinates, and a self-reported six-digit NAICS industry code. It is
refreshed continuously—the copy read while writing this notebook was updated on 8 August 2026.

**The licence is the cleanest in this whole challenge pack.** It is published under the Open Data
Commons **Public Domain Dedication and License (PDDL)**, whose text states that recipients *"may
use this work commercially, use technical protection measures, combine this data or database with
other databases or data, and share their changes and additions or keep them secret."* Commercial
use explicit, no attribution required, no share-alike. Nothing here needs a lawyer.

## The thing that makes San Francisco unusually good for this

Most cities publish business licences as a flat list and leave you to decide what counts as a
neighbourhood. San Francisco assigns each business a **`business_corridor`**—its own designation
of which commercial strip the business belongs to.

That matters more than it sounds. Drawing your own bounding box means defending it, and a box
that is slightly wrong produces a graph with no structure in it—too big and every business is
connected to nothing in particular, too small and there is no cascade to trace. **Using the
City's boundary means the geography question is answered by somebody with standing to answer
it**, and you spend your afternoon on the economics instead.

## One thing before we download

We are going to print the columns we actually received before using any of them. This is a habit
worth acquiring: a column name in someone's documentation is only half a claim, and the other
half is whether it is there today. Public datasets change without telling you.

In [ ]:
import requests
import pandas as pd

def fetch_socrata(base, where, limit=5000, retries=3):
    """Socrata with a simple backoff. 150 people hit this in the same ten minutes."""
    params = {"$where": where, "$limit": limit}
    for attempt in range(retries):
        try:
            r = requests.get(base, params=params, timeout=90)
            r.raise_for_status()
            return pd.DataFrame(r.json())
        except Exception as exc:                                   # noqa: BLE001
            if attempt == retries - 1:
                raise
            wait = 2 ** attempt
            print(f"  attempt {attempt + 1} failed ({exc}); retrying in {wait}s")
            time.sleep(wait)

with step("download registered businesses"):
    if CITY == "San Francisco":
        # location_end_date IS NULL means the business has not closed this location.
        WHERE = (f"business_corridor='{CORRIDOR}' AND location_end_date IS NULL")
    else:
        lo_lat, hi_lat, lo_lon, hi_lon = LA_BBOX
        WHERE = (f"location_end_date IS NULL")
    raw = fetch_socrata(SOCRATA, WHERE, limit=5000)

print(f"\nRows returned: {len(raw):,}")
print("\nColumns actually present:")
for c in sorted(raw.columns):
    print(f"  {c}")

**Print the columns before you trust them.** We just did, and you should look at what came back
rather than skipping to the next cell—particularly whether `self_reported_naics_code` is there,
because the whole supply layer in Section 8 hangs off it.

Now let us see how complete the industry code actually is. Roughly one business in six does not
report one, and how you handle those is a decision rather than an oversight.

In [ ]:
NAICS_COL = "self_reported_naics_code" if CITY == "San Francisco" else "naics"
NAME_COL  = "dba_name" if CITY == "San Francisco" else "business_name"

if NAICS_COL not in raw.columns:
    raise RuntimeError(
        f"Expected an industry column {NAICS_COL!r} and it is not in this response. "
        f"Columns present: {sorted(raw.columns)}. The publisher's schema has changed - "
        f"tell a coach before building on this."
    )

have_naics = raw[NAICS_COL].notna().sum()
print(f"Businesses with an industry code : {have_naics:,} of {len(raw):,} "
      f"({have_naics / max(len(raw), 1):.0%})")
print(f"Businesses with coordinates      : {raw['location'].notna().sum():,}"
      if "location" in raw.columns else "  (no location column!)")
print()
print("Most common industry codes in this corridor:")
print(raw[NAICS_COL].value_counts().head(8).to_string())

# 4. Two things we filter on sight, and why

Public data being *open* is not the same as public data being *appropriate*. This file is
published under a public-domain dedication and we are entirely within our rights to use all of
it—and we are still going to throw two categories away before we build anything, because being
allowed to do something is a different question from whether you should.

**Sole proprietors.** A great many small businesses are registered in a person's own name. San
Francisco's file has no column marking them, which means there is no single flag to drop. What
we are looking at, in practice, is a natural person's name sitting in a dataset we are about to
publish a graph of.

**Home addresses.** Related and worse. A registration at `"562 A Filbert St Apt 4"` is somebody's
apartment. Putting a residential address into a demo that a room of 150 people will look at, and
that gets promoted afterwards, is not something we want to do by accident.

Neither filter is perfect. We are using three heuristics together, and we are going to print how
many rows each one removes rather than doing it quietly:

1. Keep entities whose legal name carries a corporate suffix—`Inc`, `LLC`, `Corp`, `Ltd`, `LP`, `Company`
2. Keep storefront industry families—retail, food service, personal and repair services
3. Drop any address containing `Apt`, `Unit`, `Ste` or `#`

**This costs us businesses, and some of them are real storefronts run by sole proprietors.** That
is a genuine trade-off, not a free win, and if your team wants to argue for a different line you
should—just make it deliberately and be ready to say why.

In [ ]:
import re

CORP_SUFFIX = re.compile(r"\b(inc|llc|l\.l\.c|corp|corporation|ltd|limited|lp|llp|company|co)\b\.?",
                         re.IGNORECASE)
RESIDENTIAL = re.compile(r"\b(apt|unit|ste|suite)\b|#", re.IGNORECASE)

# Storefront NAICS families: 44-45 retail, 71 arts/rec, 72 food service,
# 81 personal & repair services, 51 information, 54 professional services.
STOREFRONT_PREFIXES = ("44", "45", "71", "72", "81", "54", "51", "62", "31", "32", "33")

ADDR_COL = "full_business_address" if CITY == "San Francisco" else "street_address"
LEGAL_COL = "ownership_name" if CITY == "San Francisco" else "business_name"

df = raw.copy()
before = len(df)

df["naics"] = df[NAICS_COL].astype(str).str.extract(r"(\d+)")[0]
df = df[df["naics"].notna()]
after_naics = len(df)

is_corp   = df[LEGAL_COL].astype(str).apply(lambda s: bool(CORP_SUFFIX.search(s)))
is_home   = df[ADDR_COL].astype(str).apply(lambda s: bool(RESIDENTIAL.search(s)))
is_store  = df["naics"].str[:2].isin(STOREFRONT_PREFIXES)

print(f"Started with                              : {before:,}")
print(f"  dropped: no usable industry code        : -{before - after_naics:,}")
print(f"  dropped: no corporate suffix in legal name: -{(~is_corp).sum():,}")
print(f"  dropped: address looks residential      : -{is_home.sum():,}")
print(f"  dropped: not a storefront industry      : -{(~is_store).sum():,}")

df = df[is_corp & (~is_home) & is_store].copy()
print(f"\nRemaining                                 : {len(df):,}")
print(f"That is {len(df) / max(before, 1):.0%} of what the City published for this corridor.")

if len(df) < 60:
    print()
    print("  WARNING - fewer than 60 businesses survived. A graph this small will")
    print("  have very little structure in it and your cascade will be trivial.")
    print("  Try a larger corridor (Section 1 lists counts) or loosen a filter")
    print("  deliberately and say so in your demo.")

Look at those numbers, because the corporate-suffix filter is doing most of the work and it is
the bluntest of the three. It removes every sole proprietor, but it also removes every
partnership and every business that simply registered under a trade name without a suffix.

**If that number looks uncomfortably large to you, you are reading it correctly.** This is the
kind of decision a judge will ask about, and "we thought about it and here is where we drew the
line" is a much better answer than either "we used everything" or silence.

In [ ]:
# Normalise into the node table we will actually build the graph on.
import json as _json

def point_of(row):
    loc = row.get("location")
    if isinstance(loc, dict) and loc.get("coordinates"):
        lon, lat = loc["coordinates"][0], loc["coordinates"][1]
        return pd.Series({"longitude": float(lon), "latitude": float(lat)})
    return pd.Series({"longitude": None, "latitude": None})

coords = df.apply(point_of, axis=1)
businesses = pd.DataFrame({
    "business_id": df.get("location_id", pd.Series(range(len(df)))).astype(str).values,
    "name":        df[NAME_COL].astype(str).str.strip().values,
    "legal_name":  df[LEGAL_COL].astype(str).str.strip().values,
    "address":     df[ADDR_COL].astype(str).str.strip().values,
    "zip":         df.get("business_zip", df.get("zip_code", pd.Series([None] * len(df)))).astype(str).str[:5].values,
    "naics":       df["naics"].values,
    "naics4":      df["naics"].str[:4].values,
    "corridor":    (CORRIDOR if CITY == "San Francisco" else "bbox"),
    "latitude":    coords["latitude"].values,
    "longitude":   coords["longitude"].values,
})
businesses = businesses[businesses.latitude.notna()].drop_duplicates("business_id")

# One more filter, and it exists because of a real incident: a single business in
# Central Market carried a name of one character, the validation section failed
# the whole corridor on it, and 355 perfectly good businesses went unpublished.
# A junk name is dirty data to drop, not grounds to reject a neighbourhood.
unusable = businesses.name.isna() | (businesses.name.str.strip().str.len() < 2)
if unusable.any():
    print(f"  dropped {int(unusable.sum())} row(s) with an unusable name")
businesses = businesses[~unusable]
businesses = businesses.reset_index(drop=True)

print(f"Node table: {len(businesses):,} businesses")
print(f"Distinct 4-digit industries: {businesses.naics4.nunique()}")
print()
print(businesses[["name", "naics4", "zip"]].head(8).to_string(index=False))

# 5. What the City does not tell you

You now have a clean list of real businesses with real locations and real industry codes. Here is
the problem, and it is the whole reason this challenge is interesting:

**Nothing in that file says any of them are connected to each other.**

There is no column for who supplies whom. No column for shared customers. No column for the
landlord, the lender, the buying co-op, the crews who eat next door. A business registry is a
list of dots, and what we need is a network.

This is not an oversight by San Francisco. **That information does not exist anywhere in United
States public data.** We looked hard: federal contract sub-awards name a prime and a
subcontractor, but only above thirty thousand dollars and rarely on the same block. Hazardous
waste manifests genuinely link a generator to a receiving facility, but only for large industrial
generators. State liquor boards record brewer-to-distributor sales—and the one with the best data
forbids commercial use by statute. Every remaining commercial supplier database is proprietary.

So the honest position is this. **We have one real relationship available to us, and we have to
model the rest.** The next four sections do exactly that, in order of how real they are, and each
one tells you plainly which it is.

# 6. The one real connection: who lent to whom

## Why this is the good one

The Small Business Administration publishes every 7(a) and 504 loan it has guaranteed, under
FOIA, as a plain CSV. Each row names **the borrowing business** and **the lending bank**, with
addresses for both. That is a genuine, measured, business-to-business relationship with real
names on both ends—the only one in this notebook.

It carries something even more useful. Alongside the loan there is `LoanStatus`, `ChargeOffDate`
and `GrossChargeOffAmount`. **A charged-off loan is a business that actually failed.**

Think about what that gives you. Every other signal in this notebook is a model of fragility.
This one is an outcome. If your cascade analysis says a particular kind of business is
structurally exposed, you can go and check whether businesses like it historically failed. Very
few teams will do this, and it is the strongest validation story available in this challenge.

The licence is stated as **"Public Domain"** on the resource page and **"U.S. Government Works"**
on the dataset page—a work of the United States government, and clean.

## Two things about the file

**It is large.** The FY2020-to-present 7(a) file is about 144 MB, and we need a fraction of a
percent of it. So we read it in chunks and keep only the rows for our state, rather than pulling
the whole thing into memory. That is a habit worth having—the cost of a careless `read_csv` on a
public dataset is usually paid by whoever runs your notebook after you.

**We filter out individual borrowers, deliberately.** The file has a `BusinessType` column taking
values Individual, Partnership and Corporation. Rows marked Individual are sole proprietors whose
`BorrName` is a person's name—the same concern as Section 4, and here there is an actual column
to filter on rather than a heuristic. We use it.

In [ ]:
SBA_URL = ("https://data.sba.gov/sites/default/files/uploaded_resources/"
           "FOIA_7a_FY2020_Present_asof_260630.csv")

SBA_COLS = ["BorrName", "BorrStreet", "BorrCity", "BorrState", "BorrZip",
            "BankName", "BankFDICNumber", "BankState",
            "GrossApproval", "ApprovalDate", "TermInMonths", "NaicsCode",
            "FranchiseCode", "FranchiseName", "BusinessType", "BusinessAge",
            "LoanStatus", "ChargeOffDate", "GrossChargeOffAmount", "JobsSupported"]

sba = None
with step("download SBA loan records"):
    try:
        keep = []
        reader = pd.read_csv(SBA_URL, chunksize=100_000, low_memory=False,
                             encoding="latin-1", on_bad_lines="skip")
        for i, chunk in enumerate(reader):
            cols = [c for c in SBA_COLS if c in chunk.columns]
            if i == 0:
                missing = [c for c in SBA_COLS if c not in chunk.columns]
                print(f"  columns present: {len(cols)} of {len(SBA_COLS)}")
                if missing:
                    print(f"  NOT present (schema has changed): {missing}")
            sub = chunk[cols]
            if "BorrState" in sub.columns:
                sub = sub[sub.BorrState.astype(str).str.strip().str.upper() == STATE]
            keep.append(sub)
        sba = pd.concat(keep, ignore_index=True)
        print(f"\n  rows for {STATE}: {len(sba):,}")
    except Exception as exc:                                        # noqa: BLE001
        print(f"\n  COULD NOT REACH data.sba.gov: {exc}")
        print("  This layer is optional - the notebook continues without it, but you")
        print("  lose the only fully-real edge and the charge-off outcomes. Tell a coach.")
        sba = pd.DataFrame(columns=SBA_COLS)

Now the filter we said we would apply, and then a look at what is actually in there.

In [ ]:
if len(sba):
    before = len(sba)
    if "BusinessType" in sba.columns:
        sba = sba[sba.BusinessType.astype(str).str.strip().str.lower() != "individual"]
    print(f"Dropped individual (sole proprietor) borrowers: {before - len(sba):,}")
    print(f"Remaining {STATE} loans: {len(sba):,}\n")

    if "LoanStatus" in sba.columns:
        print("Loan outcomes in this state:")
        print(sba.LoanStatus.value_counts().head(6).to_string())
    if "ChargeOffDate" in sba.columns:
        failed = sba.ChargeOffDate.notna().sum()
        print(f"\nLoans charged off (the business failed): {failed:,} "
              f"({failed / max(len(sba), 1):.1%})")
        print("That percentage is a real base rate for small business failure among")
        print("SBA borrowers in this state. It is a number worth putting in your demo.")
    if "FranchiseName" in sba.columns:
        fr = sba.FranchiseName.notna().sum()
        print(f"\nLoans to franchised businesses: {fr:,} - each one a real "
              f"business-to-brand edge")
else:
    print("No SBA data loaded - skipping.")

## Joining it to our corridor, and being honest about the join

Now the awkward part. The SBA file identifies a business by name and street address. Our corridor
identifies a business by name and street address. Neither has an identifier the other shares, so
we have to match on text—and text matching between two government files is never clean.

We normalise both names hard (upper case, strip punctuation and corporate suffixes) and require
the ZIP code to agree. **Expect a low match rate.** Most businesses have never taken an SBA loan,
and of those that have, many registered under a different name than they borrowed under.

**Report both numbers—rows before the join and rows after.** A join that ate your data looks
exactly like "there is no data here," and one count cannot tell the difference.

In [ ]:
def norm_name(s):
    s = str(s).upper()
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    s = CORP_SUFFIX.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()

loan_edges = pd.DataFrame(columns=["business_id", "bank_name", "gross_approval",
                                   "approval_date", "loan_status", "charged_off"])
if len(sba) and {"BorrName", "BorrZip"}.issubset(sba.columns):
    # We match on NAME ONLY, restricted to the city's ZIP prefix - deliberately,
    # and this is worth reading because it is a data lesson rather than a code
    # detail.
    #
    # The obvious join is name AND ZIP. It returns nothing, and the reason is
    # that `business_zip` in the city file is NOT the ZIP the business sits in.
    # This corridor's businesses came back carrying five different ZIPs spread
    # across San Francisco while their COORDINATES sit inside eight blocks -
    # because the registry records a mailing address, which for many small
    # businesses is an accountant or a home.
    #
    # The coordinates are trustworthy. The ZIP is not. Joining on a field
    # because it exists and looks like the right shape is how you get zero rows
    # and no explanation.
    CITY_ZIP_PREFIX = "941"      # San Francisco

    b = businesses.assign(_k=businesses.name.map(norm_name))
    b_legal = businesses.assign(_k=businesses.legal_name.map(norm_name))
    b_all = pd.concat([b, b_legal]).drop_duplicates(subset=["business_id", "_k"])
    b_all = b_all[b_all._k.str.len() > 3]

    s = sba.assign(_k=sba.BorrName.map(norm_name),
                   _z=sba.BorrZip.astype(str).str.extract(r"(\d{5})")[0])
    s = s[s._k.str.len() > 3]
    s_city = s[s._z.astype(str).str.startswith(CITY_ZIP_PREFIX)]

    merged = b_all.merge(s_city, on="_k", how="inner", suffixes=("", "_sba"))
    # Report every count around the join. One number cannot tell "no data here"
    # apart from "the join ate it".
    print(f"Businesses in corridor        : {len(businesses):,}")
    print(f"SBA loans in state            : {len(s):,}")
    print(f"  ...in this city             : {len(s_city):,}")
    print(f"Matched loan records          : {len(merged):,}")
    print(f"Distinct businesses matched   : {merged.business_id.nunique():,} "
          f"({merged.business_id.nunique() / max(len(businesses), 1):.1%} of the corridor)")

    if len(merged):
        loan_edges = pd.DataFrame({
            "business_id":    merged.business_id.values,
            "bank_name":      merged.BankName.astype(str).values,
            "gross_approval": pd.to_numeric(merged.GrossApproval, errors="coerce").values,
            "approval_date":  merged.ApprovalDate.astype(str).values,
            "loan_status":    merged.get("LoanStatus", pd.Series([None] * len(merged))).astype(str).values,
            "charged_off":    merged.get("ChargeOffDate", pd.Series([None] * len(merged))).notna().values,
        })
print()
print("A low match rate here is expected and is not a bug. Most businesses have")
print("never borrowed from the SBA. What matters is that the ones that did carry a")
print("REAL lender relationship and, sometimes, a REAL failure.")

# 7. Modelling the supply edge, honestly

## Why we are about to do something that looks like cheating, and why it is not

We need to know which businesses buy from which. That information does not exist. So we are
going to generate those edges—and the entire question is whether we generate them from
*imagination* or from *measurement*.

Here is the distinction, and it is worth reading twice because it is the sentence you will be
asked about:

> The Bureau of Economic Analysis publishes, from actual survey and tax data, **how much every
> industry buys from every other industry**. That a restaurant spends a measurable share of its
> input budget with food wholesalers is a **published federal statistic**, not a guess.
>
> That *this* restaurant buys from *that* wholesaler is **ours**. A modelling choice.

So the **type** of relationship and its **average intensity** are real. The **pairing** is not.
Everything downstream inherits that split, and saying so is not a disclaimer—it is the accurate
description of what you have.

## What we are loading

What we want is a **direct requirements** coefficient for each pair of industries: the cents of
input industry A buys from industry B **per dollar of A's own output**. We use the **summary
level**, roughly seventy industries, which corresponds to about four-digit NAICS—which is why the
node table carries a `naics4` column.

**BEA does not publish that table through its API**, and the reason we did not simply use the one
it does publish is worth understanding, because it is the same idea the whole challenge rests on.

BEA offers **total requirements**: everything industry B must produce, economy-wide, to deliver
one dollar of A to final demand. That number already includes every indirect round—A buys from B,
B buys from C, C buys from B again, and so on, summed to convergence.

**That is exactly what your graph traversal is going to compute.** Put total requirements on the
edges and then walk three hops, and you count the indirect effects twice: once inside each
coefficient, once along the path. The cascade totals come out inflated, and—worse—they come out
inflated *plausibly*, which is the kind of wrong that survives a demo.

So we compute direct requirements ourselves, the standard way, from BEA's Use table:

$$A_{ij} = \frac{\text{Use}_{ij}}{X_j}$$

where `Use[i,j]` is the dollars of commodity *i* consumed by industry *j*, and `X[j]` is
industry *j*'s total output. Divide the column by its own total and you get shares of a dollar.
`scripts/fetch_bea.py` does this and writes the result; the maths is four lines and it is worth
reading.

**Edges carry the direct effect. The graph supplies the indirect one.** That division of labour is
the reason a graph is the right tool here rather than a bigger table.

Why summary rather than the finer detail table: detail exists only for benchmark years, summary
is updated annually, and a corridor of a few hundred businesses will contain nowhere near four
hundred distinct industries anyway. Finer granularity would buy sparsity, not precision.

**Why the file is in the repo rather than downloaded.** BEA's API requires a free registration
key. That is fine for one person and a bad idea for a room of a hundred and fifty on event
morning, so we fetched it once and pinned it. The generator that produced it is in
`scripts/`, so you can regenerate or inspect it—you should not have to trust a table you cannot
audit.

In [ ]:
# The pinned BEA coefficients travel with the repo. If your team forked the repo,
# this raw URL still points at the upstream copy, which is what you want.
REPO_RAW = ("https://raw.githubusercontent.com/haggman/"
            "A4I2026-challenge-3-micro-grants/main/")
BEA_URL = REPO_RAW + "data/bea_direct_requirements.csv"

with step("load BEA direct requirements"):
    try:
        bea = pd.read_csv(BEA_URL, dtype=str)
        bea["coefficient"] = pd.to_numeric(bea["coefficient"], errors="coerce")
        bea = bea[bea.coefficient.notna() & (bea.coefficient > 0)]
        print(f"  {len(bea):,} industry-pair coefficients")
        print(f"  {bea.buyer_naics4.nunique()} buying industries, "
              f"{bea.supplier_naics4.nunique()} supplying industries")
    except Exception as exc:                                        # noqa: BLE001
        raise RuntimeError(
            f"Could not load the pinned BEA coefficients from {BEA_URL}\n"
            f"  ({exc})\n"
            f"If you are a maintainer and this file does not exist yet, run "
            f"scripts/fetch_bea.py with a free BEA API key to generate it."
        ) from exc

print()
print("The strongest purchase relationships in the table:")
print(bea.nlargest(6, "coefficient")[
    ["buyer_naics4", "supplier_naics4", "coefficient"]].to_string(index=False))

Now the allocation rule, and notice how boring it is on purpose.

For every pair of businesses in the corridor, we look up whether BEA says the buyer's industry
purchases from the supplier's industry. If it does, and the coefficient clears a floor, we create
a `Supplies` edge and carry the coefficient onto it as a weight. We also require the two
businesses to be within a walkable distance of each other, because the whole premise is a local
economy.

**We deliberately did not make this clever.** No randomness, no hidden scoring, no tuning to make
the demo look good. Every edge is reproducible from data you can see, which means you can inspect
it, argue with it, and improve it—and improving this rule is one of the best add-ons available
in this challenge. A team that replaces distance-and-coefficient with something better, and can
say why it is better, has done real work.

In [ ]:
import numpy as np

SUPPLY_COEFF_FLOOR = 0.005      # 0.5 cents of input per dollar of output
MAX_SUPPLY_METRES  = 1200       # a walkable corridor, not a metro

# The two constants that decide whether your cascade means anything.
#
# Without them this corridor produced 5,515 edges across 148 businesses - a
# quarter of every possible pair - and a two-hop traversal reached essentially
# everything from everywhere. A cascade that always answers "most of the
# corridor" is not an answer.
#
# GENERIC_SUPPLIER_PCTL drops the most ubiquitous input industries. Every
# business buys accounting, legal and advertising, so those firms end up
# supplying almost the whole corridor - and if your accountant closes you hire
# another one on Monday. Ubiquitous inputs are substitutable, and substitutable
# is the opposite of what a cascade is about.
#
# MAX_SUPPLIERS_PER_BUYER keeps only each business's strongest few supplier
# relationships. A business has a handful of inputs that would actually hurt to
# lose, not forty.
GENERIC_SUPPLIER_PCTL   = 0.90
MAX_SUPPLIERS_PER_BUYER = 5

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp = p2 - p1
    dl = np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def naics_keys(code):
    """Every prefix of a NAICS code, 2 through 6 digits.

    BEA's industry codes do not sit at one NAICS depth. A summary code like
    311FT maps to '311' - three digits - while another maps to '7225'. Our
    businesses carry six-digit codes truncated to four. Matching those on
    equality would join almost nothing, and the symptom would be an empty
    supply table rather than an error, which is the worst kind.

    So we expand each business into all of its prefixes and join on those. A
    restaurant with NAICS 722511 matches a BEA row keyed '72', '722' or '7225'.
    """
    code = re.sub(r"\D", "", str(code))
    return [code[:n] for n in range(2, min(len(code), 6) + 1)] or [code]

with step("build supply edges"):
    pairs = bea[bea.coefficient >= SUPPLY_COEFF_FLOOR][
        ["buyer_naics4", "supplier_naics4", "coefficient"]]

    keyed = businesses[["business_id", "naics", "latitude", "longitude"]].copy()
    keyed["key"] = keyed.naics.map(naics_keys)
    keyed = keyed.explode("key")
    print(f"  {len(businesses):,} businesses -> {len(keyed):,} NAICS prefix keys")

    buyers    = keyed.rename(columns={"business_id": "buyer_id", "key": "buyer_naics4",
                                      "latitude": "blat", "longitude": "blon"})
    suppliers = keyed.rename(columns={"business_id": "supplier_id", "key": "supplier_naics4",
                                      "latitude": "slat", "longitude": "slon"})

    # Drop ubiquitous input industries before pairing anything up.
    reach = pairs.groupby("supplier_naics4").buyer_naics4.nunique()
    cutoff = reach.quantile(GENERIC_SUPPLIER_PCTL)
    generic = set(reach[reach > cutoff].index)
    print(f"  dropping {len(generic)} ubiquitous supplier industries "
          f"(serve more than {cutoff:.0f} buying industries)")
    pairs = pairs[~pairs.supplier_naics4.isin(generic)]

    cand = buyers.merge(pairs, on="buyer_naics4").merge(suppliers, on="supplier_naics4")
    # A pair can match at several prefix depths. Keep the strongest coefficient
    # per business pair rather than one edge per depth.
    cand = (cand.sort_values("coefficient", ascending=False)
                .drop_duplicates(subset=["buyer_id", "supplier_id"]))
    cand = cand[cand.buyer_id != cand.supplier_id]
    print(f"  candidate pairs from BEA + industry mix: {len(cand):,}")

    cand["metres"] = haversine_m(cand.blat.values, cand.blon.values,
                                 cand.slat.values, cand.slon.values)
    cand = cand[cand.metres <= MAX_SUPPLY_METRES]
    print(f"  within {MAX_SUPPLY_METRES} m: {len(cand):,}")

    # Keep each buyer's strongest few supplier relationships, nearest first on a
    # tie. This is the step that turns a hairball into a graph you can traverse.
    supply_edges = (cand.sort_values(["buyer_id", "coefficient", "metres"],
                                     ascending=[True, False, True])
                        .groupby("buyer_id", sort=False)
                        .head(MAX_SUPPLIERS_PER_BUYER)
                        .copy())
    print(f"  after keeping top {MAX_SUPPLIERS_PER_BUYER} suppliers per buyer: "
          f"{len(supply_edges):,}")

    supply_edges = supply_edges[
        ["supplier_id", "buyer_id", "coefficient", "metres"]].rename(
        columns={"coefficient": "intensity"})
    supply_edges["edge_id"] = (supply_edges.supplier_id.astype(str) + "|" +
                               supply_edges.buyer_id.astype(str))
    supply_edges = supply_edges.drop_duplicates("edge_id").reset_index(drop=True)

print(f"\nSupply edges: {len(supply_edges):,}")
print(f"Businesses with at least one buyer downstream: "
      f"{supply_edges.supplier_id.nunique():,}")

Look at that second number, because it is the one that decides whether this challenge has a
cascade in it at all. If only a handful of businesses supply anyone, most of your graph is
isolated dots and the traversal has nothing to walk.

If it looks thin, the two levers are `SUPPLY_COEFF_FLOOR` and `MAX_SUPPLY_METRES` at the top of
that cell. **Changing them is legitimate and you should say you did.** Lowering the floor admits
weaker economic relationships; widening the radius admits less local ones. Both make the graph
denser and both make it less defensible. That trade-off is yours to make and to justify.

# 8. Modelling the footfall edge

## The second relationship, and a better-measured one

A lunch counter's weekday trade is the people who work within a few minutes' walk. That is not a
supply chain—no money moves between the businesses—but it is absolutely a dependency, and it is
the one a closure announcement destroys fastest. When an employer leaves, the businesses that
served its staff find out immediately.

The Census Bureau's **LEHD** program publishes exactly the number we need. The Workplace Area
Characteristics file gives, for every census block in the country, **how many people work
there**, broken down into twenty industry sectors. So "one thousand four hundred people work in
this block, of whom six hundred are in professional services" is a published federal statistic.

Again, the split: **the job counts are real, the assignment of those workers to specific
businesses is ours.**

One honest caveat that belongs in your demo if you use this: LEHD applies statistical noise to
protect confidentiality, so small counts are approximate by design. It is the right order of
magnitude and the wrong instrument for a precise claim.

## First we need to know which census blocks our corridor sits in

Block identifiers are fifteen characters, and the **first eleven are the census tract**. That is
convenient—it means we can find our tracts once and filter the LODES file by string prefix,
without needing a separate crosswalk.

Notice what we do *not* do below: hardcode a table name. Public datasets get renamed and
re-versioned, and a table that existed when this was written may not be the newest one now. We
ask BigQuery what exists and print what we found.

In [ ]:
def newest_table(dataset_path, contains):
    """Ask INFORMATION_SCHEMA rather than trusting a documented name."""
    sql = f"""
    SELECT table_name FROM `{dataset_path}`.INFORMATION_SCHEMA.TABLES
    WHERE table_name LIKE '%{contains}%' ORDER BY table_name DESC
    """
    names = [r.table_name for r in client.query(sql).result()]
    if not names:
        raise RuntimeError(f"No table matching '%{contains}%' in {dataset_path}. "
                           f"The public dataset has changed - tell a coach.")
    return names, names[0]

lat_lo, lat_hi = businesses.latitude.min(), businesses.latitude.max()
lon_lo, lon_hi = businesses.longitude.min(), businesses.longitude.max()
pad = 0.004
print(f"Corridor extent: lat {lat_lo:.4f}..{lat_hi:.4f}  lon {lon_lo:.4f}..{lon_hi:.4f}")

TRACT_TABLE = "bigquery-public-data.geo_census_tracts.us_census_tracts_national"
tract_sql = f"""
SELECT geo_id, state_fips_code,
       SAFE_CAST(internal_point_lat AS FLOAT64) AS lat,
       SAFE_CAST(internal_point_lon AS FLOAT64) AS lon
FROM `{TRACT_TABLE}`
WHERE state_fips_code = '{STATE_FIPS}'
  AND SAFE_CAST(internal_point_lat AS FLOAT64)
      BETWEEN {lat_lo - pad} AND {lat_hi + pad}
  AND SAFE_CAST(internal_point_lon AS FLOAT64)
      BETWEEN {lon_lo - pad} AND {lon_hi + pad}
"""
with step("find corridor census tracts"):
    job = client.query(tract_sql)
    tracts_df = job.to_dataframe()
    TRACT_BYTES = job.total_bytes_processed
CORRIDOR_TRACTS = set(tracts_df.geo_id.astype(str))
print(f"\nCensus tracts overlapping the corridor: {len(CORRIDOR_TRACTS)}")
print(f"Example: {sorted(CORRIDOR_TRACTS)[:3]}")
print()
print("Note internal_point_lat is a STRING in this table, not a number - the")
print("SAFE_CAST above is not decoration. Compare it to a number without casting")
print("and the query fails; sort it as a string and 9 sorts above 10 silently.")

In [ ]:
LODES_URL = (f"https://lehd.ces.census.gov/data/lodes/LODES8/"
             f"{STATE.lower()}/wac/{STATE.lower()}_wac_S000_JT00_2023.csv.gz")

# CNS01-CNS20 are NAICS 2-digit sectors. The four that actually drive daytime
# footfall on a commercial corridor:
FOOTFALL_SECTORS = {
    "CNS12": "54 professional services",
    "CNS16": "62 health care",
    "CNS15": "61 education",
    "CNS10": "52 finance and insurance",
}

wac = None
with step("download LEHD workplace jobs"):
    try:
        wac = pd.read_csv(LODES_URL, compression="gzip", dtype={"w_geocode": str})
        print(f"  {len(wac):,} workplace blocks statewide")
        wac["tract"] = wac.w_geocode.str[:11]
        wac = wac[wac.tract.isin(CORRIDOR_TRACTS)]
        print(f"  blocks in our tracts: {len(wac):,}")
    except Exception as exc:                                        # noqa: BLE001
        print(f"  COULD NOT REACH lehd.ces.census.gov: {exc}")
        print("  Continuing without the footfall layer. Tell a coach.")
        wac = pd.DataFrame()

if len(wac):
    total_jobs = int(wac["C000"].sum())
    print(f"\nTotal jobs in the corridor's tracts: {total_jobs:,}")
    print("\nBy sector (the ones that generate lunchtime and after-work trade):")
    for col, label in FOOTFALL_SECTORS.items():
        if col in wac.columns:
            print(f"  {label:<28} {int(wac[col].sum()):>8,}")

## Turning job counts into edges

A workplace block with two thousand professional-services jobs in it is a footfall source. The
businesses near it that serve daytime trade—food service, personal services, retail—are the
things those workers spend money at.

So: for each block, find the nearby consumer-facing businesses, and create a
`DrawsFootfallFrom` edge weighted by the block's job count divided across them. The block becomes
a node in its own right, which is deliberate—**an employer block is exactly the kind of thing
that "closes" in this challenge's scenario**, and a graph where the shock cannot be represented
is not much use.

In [ ]:
FOOTFALL_METRES = 500       # a five-minute walk
CONSUMER_PREFIXES = ("44", "45", "72", "81", "71")   # retail, food, personal services, arts

footfall_edges = pd.DataFrame(columns=["block_id", "business_id", "jobs", "metres"])
blocks = pd.DataFrame(columns=["block_id", "tract", "jobs", "lat", "lon"])

if len(wac) and len(CORRIDOR_TRACTS):
    # Block centroids: LODES does not ship coordinates, so we approximate each
    # block by its tract's internal point. That is coarse, and we say so.
    tract_pt = tracts_df.set_index("geo_id")[["lat", "lon"]].to_dict("index")
    blocks = pd.DataFrame({
        "block_id": wac.w_geocode.values,
        "tract":    wac.tract.values,
        "jobs":     wac["C000"].astype(int).values,
    })
    blocks["lat"] = blocks.tract.map(lambda t: tract_pt.get(t, {}).get("lat"))
    blocks["lon"] = blocks.tract.map(lambda t: tract_pt.get(t, {}).get("lon"))
    blocks = blocks[blocks.lat.notna() & (blocks.jobs > 0)].reset_index(drop=True)

    consumer = businesses[businesses.naics4.str[:2].isin(CONSUMER_PREFIXES)]
    print(f"Employer blocks with jobs   : {len(blocks):,}")
    print(f"Consumer-facing businesses  : {len(consumer):,}")

    if len(blocks) and len(consumer):
        cross = blocks.assign(_j=1).merge(consumer.assign(_j=1), on="_j")
        cross["metres"] = haversine_m(cross.lat.values, cross.lon.values,
                                      cross.latitude.values, cross.longitude.values)
        near = cross[cross.metres <= FOOTFALL_METRES].copy()
        share = near.groupby("block_id").business_id.transform("size")
        # Do NOT round here. A block with few jobs spread over many nearby
        # businesses gives shares below 0.05, and rounding those to one decimal
        # turns them into 0.0 - a weightless edge that still counts as an edge.
        near["jobs_share"] = near.jobs / share
        near = near[near.jobs_share > 0]
        footfall_edges = near[["block_id", "business_id", "jobs_share", "metres"]].rename(
            columns={"jobs_share": "jobs"})
        footfall_edges["edge_id"] = (footfall_edges.block_id.astype(str) + "|" +
                                     footfall_edges.business_id.astype(str))
        footfall_edges = footfall_edges.drop_duplicates("edge_id").reset_index(drop=True)

print(f"\nFootfall edges: {len(footfall_edges):,}")
print()
print("A note on precision you should carry into your demo: LODES does not publish")
print("block coordinates, so we place each block at its tract's internal point.")
print("Inside a dense corridor that is a few hundred metres of error. It is fine for")
print("'these workers are near these shops' and NOT fine for anything finer.")

# 9. Who lives around them

A grant decision is not only about businesses. A corridor sits in a neighbourhood, and whether
that neighbourhood is one where a closed storefront is quickly replaced or one where it stays
boarded for three years is a real difference that belongs in the argument.

We pull three measures for each census tract the corridor touches, from the American Community
Survey in BigQuery public datasets: poverty rate, vehicle access, and public-assistance receipt.
All three are real, all three are federal, and all three are aggregate—no individual records.

## Three quirks in this query, and the first one has cost us a full day before

**One: the tract identifier loses its leading zero.** This is not hypothetical. ACS stores
`geo_id` for a California tract as ten characters starting `6`, while the geometry table stores
eleven characters starting `06`. A plain equality join between them returns **zero rows**, every
downstream number comes back null, and nothing raises an error to tell you why.

It affects exactly the states with a FIPS code below 10: Alabama, Alaska, Arizona, Arkansas,
**California**, Colorado, Connecticut. San Francisco is `06`, so **this challenge is inside the
affected set by default**—which is deliberate. A bug that only appears in some states is a bug
you ship, unless your default case is one of them.

The fix is `LPAD(geo_id, 11, '0')`, which does nothing when the id is already eleven characters
and saves you when it is ten. Use it every time, forever.

**Two: there is no poverty rate column.** ACS gives you a count and its denominator, and you
divide them yourself. `SAFE_DIVIDE`, not `/`, because some tracts have a zero denominator.

**Three: we ask what tables exist rather than naming one.** The newest ACS tract table is not
the one you would guess from the year, and a previous challenge in this pack lost a test cycle
to a table name that had never existed.

In [ ]:
acs_names, ACS_TABLE = newest_table("bigquery-public-data.census_bureau_acs", "censustract")
print("ACS census-tract tables that exist right now:")
for n in acs_names[:6]:
    print(f"  {n}")
print(f"\nUsing the newest: {ACS_TABLE}")

tract_list = ",".join(f"'{t}'" for t in sorted(CORRIDOR_TRACTS))
acs_sql = f"""
SELECT
  LPAD(a.geo_id, 11, '0')                                      AS geo_id,
  SAFE_CAST(a.total_pop AS FLOAT64)                            AS total_pop,
  SAFE_CAST(a.median_income AS FLOAT64)                        AS median_income,
  SAFE_DIVIDE(SAFE_CAST(a.poverty AS FLOAT64),
              SAFE_CAST(a.pop_determined_poverty_status AS FLOAT64)) AS poverty_rate,
  SAFE_DIVIDE(SAFE_CAST(a.no_cars AS FLOAT64),
              SAFE_CAST(a.households AS FLOAT64))              AS no_vehicle_rate,
  SAFE_DIVIDE(SAFE_CAST(a.households_public_asst_or_food_stamps AS FLOAT64),
              SAFE_CAST(a.households AS FLOAT64))              AS assistance_rate
FROM `bigquery-public-data.census_bureau_acs.{ACS_TABLE}` a
WHERE LPAD(a.geo_id, 11, '0') IN ({tract_list})
"""
with step("pull tract demographics"):
    job = client.query(acs_sql)
    tract_demographics = job.to_dataframe()
    ACS_BYTES = job.total_bytes_processed

print(f"\nTracts requested from geometry : {len(CORRIDOR_TRACTS)}")
print(f"Tracts returned by ACS         : {len(tract_demographics)}")
kept = len(tract_demographics) / max(len(CORRIDOR_TRACTS), 1)
print(f"Join kept                      : {kept:.0%}")
if kept < 0.5:
    raise RuntimeError(
        f"The ACS join kept only {kept:.0%} of tracts. That is the leading-zero "
        f"failure described above, or a vintage mismatch. Do not build on this."
    )
print()
print("TWO counts, not one. A single count of zero looks exactly like 'there is no")
print("data for this area'. Two counts tell you a join ate your data.")
print()
print(tract_demographics[["geo_id", "poverty_rate", "no_vehicle_rate"]]
      .head(5).to_string(index=False))

# 10. One thing we deliberately excluded, and why

Every challenge in this pack excludes race and ethnicity as a **model input**, and requires them
instead as an **audit of the output**. This is consistent with Google's own responsible-AI
guidance, and it is worth understanding rather than just complying with.

The reasoning: race genuinely does correlate with which commercial corridors get disinvested, and
pretending otherwise would be dishonest—anyone who knows the literature on redlining will correct
you. But the correlation is a **proxy**. The causal variables here are economic and structural:
capital access, property ownership, tenure, the density of competing businesses. Those we can
measure directly, and several of them are in your data.

**And removing the column does not remove the bias.** Correlated proxies remain—ZIP code alone
carries most of the signal. This is called "fairness through unawareness" and it does not work.
The remedy is auditing the output, not deleting the input.

## But your challenge has a sharper version of this problem

Most challenges in this pack take demographics as a feature and ask you not to. **Yours has a
structural bias that has nothing to do with demographics at all, and it is worse.**

Look again at how the supply edges got built. A business gets edges when BEA says its industry
buys from another industry present in the corridor. **So businesses in well-represented,
input-heavy industries accumulate edges, and businesses in unusual or service-light industries
accumulate none.** A barber shop buys almost nothing from anyone. A restaurant buys from
everyone.

Your cascade will therefore favour restaurants and light manufacturing over personal services,
**not because they matter more to the neighbourhood but because our edge generator can see them
better.** That is a real, mechanical, structural bias in the data we handed you, and we would
rather tell you than have a judge find it.

## What auditing the outcome means here, concretely

Three steps, about twenty minutes, and most teams will skip them.

1. **Run your allocator** across the corridor and collect the businesses it would fund.
2. **Look at what industries they are in**, and compare to the industry mix of the corridor as a
   whole. Are you funding restaurants at three times their share?
3. **Look at the tracts they sit in**, and compare poverty and vehicle-access rates to the
   corridor average.

Then say the answer out loud: **did the money go where the need is, or where our edge generator
happened to have the most data?**

Both answers are worth having. If your recommendations skew toward high-need tracts, say so with
numbers. If they skew toward industries with dense supply chains, **you have found the bias we
just described, in your own output, and reporting it honestly will land better than a slide
claiming everything worked.**

# 11. Load into BigQuery

Five tables. Two are node tables, three are edge tables, and that division is the thing to hold
onto—it is exactly the shape `CREATE PROPERTY GRAPH` expects in Section 13.

| Table | Role | How real |
|---|---|---|
| `businesses` | node | **real** |
| `employer_blocks` | node | **real** job counts, approximate location |
| `supplies` | edge | rate real, pairing modelled |
| `draws_footfall_from` | edge | job counts real, pairing modelled |
| `borrowed_from` | edge | **fully real**, including failures |

Plus `tract_demographics` as an attribute table.

In [ ]:
def load_table(df, table_name, description=""):
    if not len(df):
        print(f"  {table_name:<22} SKIPPED (no rows)")
        return
    ref = f"{PROJECT_ID}.{DATASET}.{table_name}"
    job = client.load_table_from_dataframe(
        df, ref, job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
    job.result()
    if description:
        t = client.get_table(ref)
        t.description = description
        client.update_table(t, ["description"])
    print(f"  {table_name:<22} {len(df):>7,} rows")

with step("load tables into BigQuery"):
    load_table(businesses, "businesses",
               "Real registered businesses in the corridor. Source: city open data, PDDL.")
    load_table(blocks[["block_id", "tract", "jobs", "lat", "lon"]] if len(blocks) else blocks,
               "employer_blocks",
               "Workplace census blocks with real LEHD job counts. Location approximated to tract.")
    load_table(supply_edges, "supplies",
               "MODELLED. BEA industry purchase intensity allocated to co-located business pairs.")
    load_table(footfall_edges, "draws_footfall_from",
               "MODELLED. LEHD workplace job counts allocated to nearby consumer businesses.")
    load_table(loan_edges, "borrowed_from",
               "REAL. SBA 7(a)/504 FOIA loan records, business to lender, including charge-offs.")
    load_table(tract_demographics, "tract_demographics",
               "Real ACS 5-year estimates for tracts overlapping the corridor.")

Let us ask BigQuery something we could not have asked before the edges existed: **which
businesses have the most downstream dependents?** This is a one-hop version of the question your
agent will ask in three hops, and it is worth seeing that the answer is already not the same as
"the biggest business."

In [ ]:
%%bigquery
SELECT b.name, b.naics4,
       COUNT(DISTINCT s.buyer_id) AS buyers_downstream,
       ROUND(SUM(s.intensity), 4) AS total_intensity
FROM `a4i_econ.supplies` s
JOIN `a4i_econ.businesses` b ON b.business_id = s.supplier_id
GROUP BY b.name, b.naics4
ORDER BY buyers_downstream DESC
LIMIT 10

That table is the challenge in miniature, one hop deep. Now imagine it three hops deep, with the
footfall edges folded in and a fifty-thousand-dollar budget attached—that is what you are
building.

**And notice what it is not.** It is a ranking, and we just spent Section 2 explaining why a
ranking is the wrong answer. A business with four direct buyers may matter less than one with a
single buyer who is itself the sole supplier to six others. **One hop cannot see that. That is
what the graph is for.**

## Now look at that list again, sceptically

Two things are usually visible in it, and both are worth your attention before you build anything
on top.

**The same business may appear twice.** When we ran North Beach we got
`Graffeo Coffee Roasting Company Inc` with 58 buyers and `Graffeo Coffee Roasting Co.` with 34.
That is **one coffee roaster, registered twice**, and our supplier-selection rule split its
customers between the two records. The real business reaches 92 businesses; the graph shows two
smaller ones, and an agent recommending either would understate what the grant protects.

Nobody is at fault here. Businesses re-register when they change entity type, move, or add a
location, and old records do not always close. **Every municipal business registry in the country
has this in it.** We have deliberately not de-duplicated, because near-duplicate matching is a
judgment call with real false-positive risk—merge two genuinely different businesses and you
invent a dependency that does not exist.

So it is yours to decide. Leave it and say so, or merge on normalised name plus address and be
able to defend the threshold you picked. Either is a good answer. Not noticing is not.

**And one supplier probably dominates.** Whichever industry has only one business in your corridor
collects every edge for that input, because it is the only local source. That is true within the
model and it is worth saying out loud rather than presenting a single chocolatier as the keystone
of the neighbourhood economy. Ask yourself whether a buyer could simply purchase from outside the
corridor—for most inputs the honest answer is yes, and that should temper how strong you claim the
dependency is.

## Where to start looking, because most businesses are leaves

Run this before you pick a business to demo. It answers the question that will otherwise cost you
twenty minutes: **which businesses actually have something downstream of them?**

The answer is: not many, and that is the point rather than a defect. Most businesses on a
commercial street sell to the public, not to each other. A hair salon supplies nobody. Trace a
cascade from one and you correctly get nothing.

**So the corridor sorts itself into three groups** — the businesses with real downstream reach,
the businesses that buy but do not supply, and the leaves. Finding the first group is not a
warm-up exercise for your agent, it is a large part of what your agent is *for*. A grant officer
with forty applications has no way to know which of them hold anything up.

In [ ]:
depth = client.query(f"""
WITH one_hop AS (
  SELECT supplier_id AS root, buyer_id AS reached, 1 AS hops
  FROM `{PROJECT_ID}.{DATASET}.supplies`
),
two_hop AS (
  SELECT a.supplier_id AS root, b.buyer_id AS reached, 2 AS hops
  FROM `{PROJECT_ID}.{DATASET}.supplies` a
  JOIN `{PROJECT_ID}.{DATASET}.supplies` b ON b.supplier_id = a.buyer_id
  WHERE b.buyer_id != a.supplier_id
),
reach AS (SELECT * FROM one_hop UNION ALL SELECT * FROM two_hop)
SELECT b.name, b.naics4,
       COUNT(DISTINCT IF(hops = 1, reached, NULL)) AS direct,
       COUNT(DISTINCT reached)                     AS within_2_hops
FROM reach r
JOIN `{PROJECT_ID}.{DATASET}.businesses` b ON b.business_id = r.root
GROUP BY b.name, b.naics4
HAVING COUNT(DISTINCT IF(hops = 2, reached, NULL)) > 0
ORDER BY within_2_hops DESC
""").to_dataframe()

print(f"Businesses with a two-hop cascade: {len(depth)} of {len(businesses)} "
      f"({len(depth) / max(len(businesses), 1):.0%})\n")
print(depth.head(12).to_string(index=False))
print()
print("Everything NOT on this list is a leaf or a one-hop supplier. That is not")
print("a data problem - it is what a commercial street looks like. But it does")
print("mean picking a demo business at random will usually show you nothing, so")
print("start here and work outward.")
print()
print("Worth asking before you build: is 'has the deepest cascade' the same as")
print("'most deserves a grant'? It is not, and the gap between those two is")
print("where your judgment shows up.")

# 12. Validate before you traverse

Every table above could be empty, malformed, or subtly wrong in a way that produces a graph which
loads perfectly and answers every question incorrectly. The checks below query the **loaded
BigQuery tables**, not the dataframes in memory, because those are what your agent will read.

**One of these checks matters more than all the others**, and it is specific to graphs.

> BigQuery's own documentation says: *"Element keys must be unique, but BigQuery doesn't perform
> any uniqueness checking. You are responsible for ensuring that your element keys are unique."*
> It also says: *"Rows with a null element key are ignored."*

Read that carefully. If a key appears twice, your graph does not error—it **silently duplicates
paths**, and every count downstream is inflated. If a key is null, the row **silently vanishes**.
Neither raises. Neither is visible in a row count.

This is Challenge 3's version of the sentinel-value trap: the errors that hurt you are the ones
that do not raise.

In [ ]:
CHECKS = []
def check(name, passed, detail=""):
    CHECKS.append((name, bool(passed), str(detail)))

def q1(sql):
    return client.query(sql).to_dataframe().iloc[0]

DS = f"{PROJECT_ID}.{DATASET}"

# ------------------------------------------------------------------ businesses
b = q1(f"""
SELECT COUNT(*) AS n,
       COUNT(DISTINCT business_id) AS ids,
       COUNTIF(business_id IS NULL) AS null_ids,
       COUNTIF(latitude IS NULL OR longitude IS NULL) AS no_coords,
       COUNT(DISTINCT naics4) AS industries,
       COUNTIF(name IS NULL OR LENGTH(name) < 2) AS bad_names
FROM `{DS}.businesses`
""")
check("businesses: has rows",        b.n >= 60,        f"{int(b.n):,} businesses")
check("businesses: keys unique",     b.ids == b.n,     f"{int(b.ids):,} distinct of {int(b.n):,}")
check("businesses: no null keys",    b.null_ids == 0,  f"{int(b.null_ids)} null ids")
check("businesses: all located",     b.no_coords == 0, f"{int(b.no_coords)} without coordinates")
check("businesses: industry spread", b.industries >= 8, f"{int(b.industries)} distinct NAICS4")
check("businesses: names usable",    b.bad_names == 0, f"{int(b.bad_names)} unusable names")

# Geography: everything must actually be in the corridor, or the cascade is fiction.
g = q1(f"""
SELECT MAX(latitude) - MIN(latitude) AS dlat,
       MAX(longitude) - MIN(longitude) AS dlon
FROM `{DS}.businesses`
""")
check("businesses: geographically tight", g.dlat < 0.12 and g.dlon < 0.12,
      f"extent {g.dlat:.3f} lat x {g.dlon:.3f} lon degrees")

# ------------------------------------------------------------------ supply edges
s = q1(f"""
SELECT COUNT(*) AS n,
       COUNT(DISTINCT edge_id) AS ids,
       COUNTIF(edge_id IS NULL) AS null_ids,
       COUNTIF(supplier_id = buyer_id) AS self_loops,
       COUNT(DISTINCT supplier_id) AS suppliers,
       MAX(metres) AS max_m,
       MIN(intensity) AS min_i
FROM `{DS}.supplies`
""")
check("supplies: has rows",          s.n > 0,          f"{int(s.n):,} edges")
check("supplies: element keys unique", s.ids == s.n,
      f"{int(s.ids):,} distinct of {int(s.n):,} - duplicates would silently multiply paths")
check("supplies: no null keys",      s.null_ids == 0,  f"{int(s.null_ids)} null edge_ids")
check("supplies: no self-loops",     s.self_loops == 0, f"{int(s.self_loops)} self-referencing")
check("supplies: multiple suppliers", s.suppliers >= 5,
      f"{int(s.suppliers)} businesses supply someone")
check("supplies: within radius",     s.max_m <= MAX_SUPPLY_METRES + 1,
      f"furthest {s.max_m:.0f} m of {MAX_SUPPLY_METRES} allowed")

# Every edge endpoint must resolve to a node, or the graph has dangling references.
d = q1(f"""
SELECT
 (SELECT COUNT(*) FROM `{DS}.supplies` s
   LEFT JOIN `{DS}.businesses` b ON b.business_id = s.supplier_id
   WHERE b.business_id IS NULL) AS bad_src,
 (SELECT COUNT(*) FROM `{DS}.supplies` s
   LEFT JOIN `{DS}.businesses` b ON b.business_id = s.buyer_id
   WHERE b.business_id IS NULL) AS bad_dst
""")
check("supplies: endpoints resolve", d.bad_src == 0 and d.bad_dst == 0,
      f"{int(d.bad_src)} bad sources, {int(d.bad_dst)} bad destinations")

# THE one that decides whether a multi-hop traversal finds anything at all.
reach = q1(f"""
WITH two_hop AS (
  SELECT a.supplier_id AS root, b.buyer_id AS reached
  FROM `{DS}.supplies` a JOIN `{DS}.supplies` b ON b.supplier_id = a.buyer_id
  WHERE b.buyer_id != a.supplier_id
)
SELECT COUNT(DISTINCT root) AS roots, COUNT(*) AS paths FROM two_hop
""")
check("supplies: two-hop paths exist", reach.roots > 0,
      f"{int(reach.roots)} businesses reach someone 2 hops out ({int(reach.paths):,} paths)")

# ------------------------------------------------------------------ footfall
if footfall_edges is not None and len(footfall_edges):
    f = q1(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT edge_id) AS ids,
           COUNTIF(jobs IS NULL OR jobs <= 0) AS bad_jobs, MAX(metres) AS max_m
    FROM `{DS}.draws_footfall_from`
    """)
    check("footfall: has rows",        f.n > 0,        f"{int(f.n):,} edges")
    check("footfall: keys unique",     f.ids == f.n,   f"{int(f.ids):,} distinct")
    check("footfall: job weights sane", f.bad_jobs == 0, f"{int(f.bad_jobs)} non-positive")
    check("footfall: within radius",   f.max_m <= FOOTFALL_METRES + 1,
          f"furthest {f.max_m:.0f} m")

# ------------------------------------------------------------------ tracts
t = q1(f"""
SELECT COUNT(*) AS n,
       COUNTIF(poverty_rate < 0 OR poverty_rate > 1) AS bad_pov,
       COUNTIF(LENGTH(geo_id) != 11) AS bad_geoid,
       COUNTIF(geo_id NOT LIKE '{STATE_FIPS}%') AS wrong_state,
       ROUND(AVG(poverty_rate), 4) AS avg_pov
FROM `{DS}.tract_demographics`
""")
check("tracts: has rows",          t.n > 0,          f"{int(t.n)} tracts")
check("tracts: rates in range",    t.bad_pov == 0,   f"mean poverty {t.avg_pov:.1%}")
check("tracts: geo_id 11 chars",   t.bad_geoid == 0,
      "leading zero survived - this is the LPAD check")
check("tracts: correct state",     t.wrong_state == 0,
      f"{int(t.wrong_state)} outside FIPS {STATE_FIPS}")

tcols = [c.name.lower() for c in client.get_table(f"{DS}.tract_demographics").schema]
banned = [c for c in tcols if any(w in c for w in
          ("black", "white", "hispanic", "asian", "race", "ethnic"))]
check("tracts: no race columns",   not banned,       banned or "none present")

# ------------------------------------------------------------------ real edges
try:
    l = q1(f"SELECT COUNT(*) AS n, COUNTIF(charged_off) AS failed FROM `{DS}.borrowed_from`")
    check("loans: table present",  True,  f"{int(l.n):,} real loan edges, {int(l.failed)} charged off")
except Exception:                                                  # noqa: BLE001
    check("loans: table present",  False, "SBA layer did not load - the only fully-real edge is missing")

In [ ]:
width = max(len(name) for name, _, _ in CHECKS)
passed = sum(1 for _, ok, _ in CHECKS if ok)

print("=" * (width + 44))
print(f"{'VALIDATION':<{width}}   RESULT   DETAIL")
print("=" * (width + 44))
for name, ok, detail in CHECKS:
    print(f"{name:<{width}}   {'PASS  ' if ok else 'FAIL >>'}  {detail}")
print("=" * (width + 44))
print(f"{passed} of {len(CHECKS)} checks passed")

failures = [n for n, ok, _ in CHECKS if not ok]
if failures:
    print("\nInvestigate before you build on this:")
    for n in failures:
        print(f"  - {n}")
else:
    print("\nAll clear. Your tables are sound. Go build the graph.")

# 13. Framing your graph

This is where we stop and you start.

## What you are building

A property graph over the tables you just loaded, and a traversal that answers: **if this
business fails, what else is exposed, and what is that worth?**

You already saw the shape in Section 2 on twelve toy businesses. Now you do it on the real
corridor, with three edge types instead of one, and with the interesting parts left in.

## The DDL shape

This is the pattern, not the answer. Adapt it—in particular, decide for yourself whether your
graph should include the footfall and loan edges, and whether `employer_blocks` deserves to be a
node.

```sql
CREATE OR REPLACE PROPERTY GRAPH a4i_econ.LocalEconomy
  NODE TABLES (
    a4i_econ.businesses
      KEY (business_id)
      LABEL Business PROPERTIES (business_id, name, naics4, zip, latitude, longitude),
    a4i_econ.employer_blocks
      KEY (block_id)
      LABEL Employer PROPERTIES (block_id, jobs)
  )
  EDGE TABLES (
    a4i_econ.supplies
      KEY (edge_id)
      SOURCE KEY (supplier_id) REFERENCES businesses (business_id)
      DESTINATION KEY (buyer_id) REFERENCES businesses (business_id)
      LABEL Supplies PROPERTIES (intensity, metres),
    a4i_econ.draws_footfall_from
      KEY (edge_id)
      SOURCE KEY (block_id) REFERENCES employer_blocks (block_id)
      DESTINATION KEY (business_id) REFERENCES businesses (business_id)
      LABEL DrawsFootfall PROPERTIES (jobs, metres)
  );
```

## The five traps, so you do not lose forty minutes to them

**1. Two different naming rules, four lines apart.** In `NODE TABLES` and `EDGE TABLES`, write
`a4i_econ.businesses`—dataset-qualified and **unbackticked**. Do *not* write
`` `your-project.a4i_econ.businesses` ``: backticks around a three-part name make it one quoted
identifier, the node gets named the whole string, and nothing can reference it. Then in
`REFERENCES`, drop the dataset entirely—`REFERENCES businesses (business_id)`.

Get either wrong and you get the same unhelpful error, pointing at the `REFERENCES` line:

```
The referenced node table 'businesses' is not defined in the property graph
```

Check the `NODE TABLES` block first. The line it blames is usually not the line that is wrong.

**2. The visualisation only draws a path wrapped in `TO_JSON`.** Google's docs are explicit:
*"The query must return graph elements in JSON format using the `TO_JSON` function."* Return
`m.name` and you get a table, with **no error and no warning**. If your `%%bigquery --graph` cell
produced a table, this is almost certainly why. Return `TO_JSON(p) AS paths`, and return a
**path** rather than separate nodes and edges—otherwise intermediate elements go missing from
the picture.

**3. The notebook visualisation caps at 2 MB of returned data.** There is no node limit
documented—the limit is bytes. A wide traversal over a few hundred businesses will exceed it and
render partially. `LIMIT` before `RETURN`, and reach for `ANY SHORTEST`, are the two fixes.

**4. Duplicate element keys corrupt your answers silently.** Covered in Section 12. If you build
your own edge table, run the duplicate check on it before you build the graph over it.

**5. Spanner Graph's documentation will poison your searches.** It uses nearly identical GQL,
looks identical, and has functions BigQuery does not (`IS_ACYCLIC`, `IS_TRAIL`,
`PROPERTY_EXISTS`). Check that the URL contains `/bigquery/`.

And if your notebook cannot render at all: `!pip install bigquery-magics==0.12.1`. Both Google's
documentation and their own codelab pin that version.

## The query shape you are heading toward

```
GRAPH a4i_econ.LocalEconomy
MATCH p = (seed:Business WHERE seed.business_id = '...')
          -[:Supplies]->{1,3}(exposed:Business)
RETURN TO_JSON(p) AS paths
```

That finds what is downstream. **It is not the answer**—it is the raw material for one. Turning
a set of paths into "fund this business, for this amount, and here is what it protects" is the
work, and it is what you are judged on.

## What is actually yours to decide

Nothing below has a right answer, and every one of them is a question a judge can ask.

- **What counts as exposure?** A business three hops away with a weak coefficient is barely
  affected. Where do you stop, and why?
- **How do you weight a path?** Multiply the intensities? Take the minimum link? Count jobs at
  the far end? These give different answers and all are defensible.
- **Do footfall and supply edges mean the same thing?** They are both dependencies and they are
  not equally severe. A supplier can be replaced in a week; a lost lunch crowd never comes back.
- **What does the money actually do?** A grant does not prevent a closure, it changes a
  probability. Say what you are assuming.
- **How do you avoid funding the hub every time?** Centrality is seductive and the most connected
  business is not automatically the best investment—it may be the most robust one.
- **And the honest one:** two of your three edge types are modelled. How much weight should a
  recommendation carry when the network it rests on is partly ours? A team that can answer that
  well is doing the most interesting thinking available in this challenge.

## What you should not do

Do not use the graph to produce a ranked list and then fund the top of it. That is Section 2 with
extra steps, and it is the failure mode this challenge is designed to expose.

In [ ]:
print(f"Project        : {PROJECT_ID}")
print(f"Dataset        : {DATASET} ({LOCATION})")
print(f"City / corridor: {CITY} / {CORRIDOR if CITY == 'San Francisco' else LA_BBOX}")
print()
print("Your tables:")
for t in client.list_tables(f"{PROJECT_ID}.{DATASET}"):
    tbl = client.get_table(t)
    print(f"  {tbl.table_id:<22} {tbl.num_rows:>7,} rows")
print()
print("Next: CREATE PROPERTY GRAPH over businesses + supplies, then traverse it.")
print("See Section 13 for the DDL shape and the five traps.")

# Appendix A: diagnostic summary

If something went wrong, run this cell and send a coach what it prints. One block beats twenty
screenshots, and it carries everything needed to tell where the problem is.

In [ ]:
print("=" * 70)
print("A4I CHALLENGE 3 - DIAGNOSTIC SUMMARY")
print("=" * 70)
print(f"Project           : {PROJECT_ID}")
print(f"Dataset           : {DATASET} ({LOCATION})")
print(f"City / corridor   : {CITY} / {CORRIDOR if CITY == 'San Francisco' else LA_BBOX}")
print(f"State / FIPS      : {STATE} / {STATE_FIPS}")
print("-" * 70)
print("DISCOVERED NAMES")
print(f"  ACS table        : {globals().get('ACS_TABLE', 'not reached')}")
print(f"  Business source  : {globals().get('SOCRATA', 'not reached')}")
print(f"  SBA source       : {globals().get('SBA_URL', 'not reached')}")
print(f"  LODES source     : {globals().get('LODES_URL', 'not reached')}")
# `if acsb` would be false for a cached query, which scans zero bytes - and
# reporting "not reached" for a step that ran perfectly is how a diagnostic
# starts lying to you. Test for None, not for truth.
for label, var in [("ACS bytes", "ACS_BYTES"), ("Tract bytes", "TRACT_BYTES")]:
    v = globals().get(var)
    if v is None:
        print(f"  {label:<16} : not reached")
    elif v == 0:
        print(f"  {label:<16} : 0.0 MB (served from BigQuery's cache)")
    else:
        print(f"  {label:<16} : {v / 1e6:.1f} MB")
print("-" * 70)
print("PIPELINE")
try:
    print(f"City rows for corridor  : {len(raw):,}")
    print(f"After privacy filters   : {len(businesses):,} "
          f"({len(businesses) / max(len(raw), 1):.0%} kept)")
    print(f"Distinct NAICS4         : {businesses.naics4.nunique()}")
    print(f"SBA rows for state      : {len(sba):,}")
    print(f"  matched to corridor   : {len(loan_edges):,}")
    print(f"  of which charged off  : {int(loan_edges.charged_off.sum()) if len(loan_edges) else 0}")
    print(f"BEA coefficient pairs   : {len(bea):,}")
    print(f"Supply edges            : {len(supply_edges):,}")
    print(f"  distinct suppliers    : {supply_edges.supplier_id.nunique():,}")
    print(f"LODES blocks in tracts  : {len(blocks):,}")
    print(f"Footfall edges          : {len(footfall_edges):,}")
    print(f"Census tracts           : {len(tract_demographics):,}")
except Exception as exc:                                            # noqa: BLE001
    print(f"A stage did not complete: {exc}")
print("-" * 70)
print("LOADED TABLES")
try:
    for t in client.list_tables(f"{PROJECT_ID}.{DATASET}"):
        tbl = client.get_table(t)
        print(f"{tbl.table_id}: {tbl.num_rows:,} rows")
        print(f"    {', '.join(f.name for f in tbl.schema)}")
except Exception as exc:                                            # noqa: BLE001
    print(f"Could not list tables: {exc}")
print("-" * 70)
print("VALIDATION")
try:
    for name, ok, detail in CHECKS:
        print(f"  {'PASS' if ok else 'FAIL'}  {name}: {detail}")
except NameError:
    print("  validation section not reached")
print("-" * 70)
print("TIMING")
for name, secs in STEP_TIMES.items():
    print(f"  {name:<34} {secs:>7.1f}s")
print(f"  {'TOTAL WALL CLOCK':<34} {time.time() - NOTEBOOK_START:>7.1f}s")
print("=" * 70)

# Appendix B: using a corridor we have not tested

Runs standalone—it needs only the config cell at the top. It does two things:

1. Lists **every** commercial corridor San Francisco publishes, with how many active businesses
   each one has and how many survive our privacy filters.
2. Tells you plainly whether a corridor is worth building on.

The number that matters is not the raw business count, it is **how many distinct four-digit
industries** the corridor contains. Supply edges only exist between industries BEA says trade
with each other, so a corridor of two hundred businesses spread across forty industries produces
a far richer graph than three hundred businesses that are all restaurants.

In [ ]:
# c3_90_publish_snapshot.ipynb skips this cell - it is a diagnostic, not part of
# the pipeline, and running it per-corridor would re-fetch the city API needlessly.
# =====================================================================
# CORRIDOR VIABILITY TESTER - standalone
# =====================================================================
import requests as _rq
import pandas as _pd

_URL = "https://data.sfgov.org/resource/g8m3-pdis.json"

print("Fetching corridor inventory from San Francisco...")
_r = _rq.get(_URL, params={
    "$select": "business_corridor, count(*) as n",
    "$where":  "location_end_date IS NULL AND business_corridor IS NOT NULL",
    "$group":  "business_corridor",
    "$order":  "n DESC",
    "$limit":  200,
}, timeout=90)
_r.raise_for_status()
_inv = _pd.DataFrame(_r.json())
_inv["n"] = _pd.to_numeric(_inv["n"])
print(f"{len(_inv)} named corridors published\n")

def assess(corridor_name, verbose=True):
    """Fetch one corridor, apply the same filters as the notebook, and judge it."""
    r = _rq.get(_URL, params={
        "$where": f"business_corridor='{corridor_name}' AND location_end_date IS NULL",
        "$limit": 5000}, timeout=90)
    r.raise_for_status()
    d = _pd.DataFrame(r.json())
    if not len(d):
        print(f"{corridor_name}: no active businesses"); return None

    total = len(d)
    d["naics"] = d.get("self_reported_naics_code", _pd.Series(dtype=str)).astype(str).str.extract(r"(\d+)")[0]
    d = d[d["naics"].notna()]
    legal = d.get("ownership_name", _pd.Series([""] * len(d))).astype(str)
    addr  = d.get("full_business_address", _pd.Series([""] * len(d))).astype(str)
    keep = (legal.str.contains(r"\b(?:inc|llc|corp|ltd|lp|llp|company|co)\b\.?", case=False, regex=True)
            & ~addr.str.contains(r"\b(?:apt|unit|ste|suite)\b|#", case=False, regex=True)
            & d["naics"].str[:2].isin(("44","45","71","72","81","54","51","62","31","32","33")))
    kept = d[keep]
    inds = kept["naics"].str[:4].nunique()

    if verbose:
        print(f"\n{corridor_name}")
        print(f"  published active      : {total:,}")
        print(f"  survives our filters  : {len(kept):,} ({len(kept)/max(total,1):.0%})")
        print(f"  distinct NAICS4       : {inds}")
        if   len(kept) < 60 or inds < 8:
            print("  VERDICT: TOO THIN. The graph will have almost no structure.")
            print("           Pick a larger corridor.")
        elif len(kept) < 120 or inds < 15:
            print("  VERDICT: WORKABLE BUT THIN. Usable, and your cascade will be")
            print("           shallow. Say so in your demo rather than hoping nobody asks.")
        elif len(kept) > 700:
            print("  VERDICT: LARGE. Good graph, but the Section 2 visualisation will")
            print("           exceed the 2 MB notebook limit - use LIMIT and ANY SHORTEST.")
        else:
            print("  VERDICT: GOOD. Enough businesses and enough industry variety.")
    return {"corridor": corridor_name, "published": total,
            "kept": len(kept), "industries": inds}

print("Ten largest corridors:")
print(_inv.head(10).to_string(index=False))
print("\n" + "=" * 62)
print("Assessing the current default plus two alternatives.")
print("Change the list below to test whichever you are considering.")
print("=" * 62)
for _c in [CORRIDOR, "24th St", "Third Street"]:
    assess(_c)